# Production-Grade RAG for Finance

- Fully local pipeline: Ollama runs the model and the embeddings, Qdrant holds the vectors, RAGWire wires it together
- Real corpus: the 2024 10-K filings of Amazon, Alphabet, and Meta
- Every finalized piece of code also lives in `scripts/`, so the notebook, the CLI, and the Chainlit app share one implementation
- LangSmith tracing is on: every model call in this session lands in the `finance-rag-bootcamp` project

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

## 1. Text extraction: PDF to markdown

- A model cannot read PDF bytes. Step one is always text
- MarkItDown converts PDF, DOCX, XLSX, PPTX to markdown-ish text
- Finance is the hard case: a 10-K is hundreds of pages of prose wrapped around numbers

In [3]:
from ragwire.loaders.markitdown_loader import MarkItDownLoader

loader = MarkItDownLoader()
doc = loader.load("data/amazon 10-k 2024.pdf")
len(doc["text_content"])

332652

In [4]:
text = doc["text_content"]
print(text[:700])

Table of Contents
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
____________________________________
FORM 10-K
____________________________________
(Mark One)
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended December 31, 2024
or
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from to .
Commission File No. 000-22513
____________________________________
AMAZON.COM, INC.
(Exact name of registrant as specified in its charter)
Delaware 91-1646860
(State or other jurisdiction of (I.R.S. Employer
incorporation or organization) Identification 


- That code is now a function in `scripts/extraction.py`, and from here on we import it

In [5]:
from scripts.extraction import pdf_to_markdown

## 2. Chunking

- The whole filing does not fit in one prompt, so we cut it into chunks
- Where you cut decides what retrieval can ever find
- This pipeline uses the page strategy, new in RAGWire 1.6: exactly one chunk per PDF page, never sub-split, merged, or overlapped
- MarkItDown flattens a PDF into one string, so the page loader reads it again with pypdf, page by page, and a retrieved chunk then maps onto a page a reader can open

In [6]:
from ragwire.loaders.page_loader import PageLoader
from ragwire.processing.splitter import PageSplitter

page_loader = PageLoader()
doc = page_loader.load("data/amazon 10-k 2024.pdf")

splitter = PageSplitter()
pages = splitter.split(doc["text_content"], pages=doc["pages"])
len(pages)

89

In [8]:
# pages

In [9]:
one = pages[41]
one["page_number"], one["page_total"]

(42, 89)

In [14]:
# print(one["text"][:600])
# print(pages[17]['text'])

- Same code, as a function, in `scripts/chunking.py`

In [15]:
from scripts.chunking import split_pages

## 3. Embeddings

- A chunk is stored as a vector: a list of numbers that captures meaning
- nomic-embed-text runs on local Ollama and produces 768 numbers per text
- Similar meaning gives similar vectors, and that is what makes search by meaning possible

In [16]:
from ragwire.embeddings.factory import get_embedding

embedder = get_embedding({"provider": "ollama", "model": "nomic-embed-text"})
vector = embedder.embed_query("What was Amazon's revenue in 2024?")
len(vector)

768

In [20]:
# vector

In [21]:
from sklearn.metrics.pairwise import cosine_similarity

v1 = embedder.embed_query("revenue increased during the year")
v2 = embedder.embed_query("net sales grew over the period")
v3 = embedder.embed_query("the cafeteria serves lunch at noon")

cosine_similarity([v1], [v2]), cosine_similarity([v1], [v3])

(array([[0.73394919]]), array([[0.32597395]]))

- Two sentences that mean the same thing score far higher than an unrelated one
- `scripts/embeddings.py` returns this same embedder

## 4. RAGWire: the pieces become one pipeline

- Everything so far was a RAGWire building block used by hand
- One config file, `config/finance_rag.yaml`, assembles them: loader, splitter, embeddings, vector store, LLM
- The commented lines in that file show the options you are not using

In [22]:
from ragwire import RAGWire

rag = RAGWire("config/finance_rag.yaml")

- During ingestion the LLM reads each filing's cover page and extracts metadata
- The instructions live in `config/metadata.yaml`, so the schema is yours to change
- Run the extractor by hand first, on the text from section 1, so ingestion is not a black box

In [23]:
len(text)

332652

In [24]:
rag.extract_metadata(text[:3000])

{'company_name': 'amazon.com, inc.',
 'doc_type': '10-k',
 'fiscal_quarter': None,
 'fiscal_year': 2024}

- Ingest one filing first and look at what it produced

In [26]:
rag.ingest_documents(["data/amazon 10-k 2024.pdf"])

Ingesting: 100%|██████████| 1/1 [00:00<00:00, 60.83file/s]


{'total': 1,
 'processed': 0,
 'skipped': 1,
 'failed': 0,
 'chunks_created': 0,
 'metadata_failed': 0,
 'replaced': 0,
 'errors': []}

In [27]:
rag.get_stats()

{'collection_name': 'finance-rag-bootcamp',
 'total_documents': 89,
 'vector_size': 768,
 'indexed': 89}

## 5. Ingesting the rest, and living with change

- Point ingestion at the whole directory. The filing we already ingested is hashed, recognized, and skipped

In [28]:
rag.ingest_directory("data")

Ingesting: 100%|██████████| 2/2 [00:47<00:00, 23.63s/file]


{'total': 2,
 'processed': 1,
 'skipped': 1,
 'failed': 0,
 'chunks_created': 108,
 'metadata_failed': 0,
 'replaced': 0,
 'errors': []}

In [29]:
rag.get_field_values(["company_name", "doc_type", "fiscal_year"])

{'company_name': ['alphabet inc.', 'amazon.com, inc.'],
 'doc_type': ['10-k'],
 'fiscal_year': [2024]}

- Filings get amended. Remove a document, or force one through again

In [30]:
rag.delete_document("data/amazon 10-k 2024.pdf")

89

In [31]:
rag.reingest_documents(["data/amazon 10-k 2024.pdf"])

Ingesting: 100%|██████████| 1/1 [00:26<00:00, 26.78s/file]


{'total': 1,
 'processed': 1,
 'skipped': 0,
 'failed': 0,
 'chunks_created': 89,
 'metadata_failed': 0,
 'replaced': 0,
 'errors': []}

## 6. Hybrid retrieval and metadata filters

- Every chunk carries two vectors: dense for meaning, sparse (BM25 style) for exact words
- Finance queries lean on exact tokens: AWS, 10-K, segment names, dollar figures
- `auto_filter` is off in this config, so `retrieve()` searches the whole collection unless you pass filters yourself

In [32]:
docs = rag.retrieve("How did AWS perform in 2024?")

for d in docs:
    print(d.metadata["file_name"], "| page", d.metadata["page_number"])

amazon 10-k 2024.pdf | page 24
amazon 10-k 2024.pdf | page 50
amazon 10-k 2024.pdf | page 25
amazon 10-k 2024.pdf | page 43


In [33]:
print(docs[0].page_content[:600])

Table of Contents
Results of Operations
We have organized our operations into three segments: North America, International, and AWS. These segments reflect the way the Company evaluates
its business performance and manages its operations. See Item 8 of Part II, “Financial Statements and Supplementary Data — Note 10 — Segment
Information.”
Overview
Macroeconomic factors, including changes in inflation and interest rates, global economic and geopolitical developments, and the development and
adoption of technologies and services, including artificial intelligence, have direct and indirect impact


In [34]:
docs = rag.retrieve("How did apple perform in 2024?")
docs

[Document(metadata={'source': 'data/amazon 10-k 2024.pdf', 'file_name': 'amazon 10-k 2024.pdf', 'file_type': 'pdf', 'file_hash': '5b582c59f1d1fa81d5112c9e0b9aa87afdbc9f42418daace257bc360e0f61ef7', 'chunk_id': '5b582c59f1d1fa81d5112c9e0b9aa87afdbc9f42418daace257bc360e0f61ef7_23', 'chunk_hash': '1a9a8c3682c5a1230c5e01255d40a948b82656c11a704ddedcced2a0c4ae44f3', 'content_hash': '69a97541f58c2e09ad4b86f8e20f573cbf256b952f48763ce96d5c1254727639', 'chunk_index': 23, 'total_chunks': 89, 'created_at': '2026-08-14T16:07:34.719964+00:00', 'metadata_status': 'ok', 'company_name': 'amazon.com, inc.', 'doc_type': '10-k', 'fiscal_quarter': None, 'fiscal_year': 2024, 'page_number': 24, 'page_total': 89, '_id': '42b53513-ece3-4db6-a6b0-1b5d6c1fb7ed', '_collection_name': 'finance-rag-bootcamp'}, page_content='Table of Contents\nResults of Operations\nWe have organized our operations into three segments: North America, International, and AWS. These segments reflect the way the Company evaluates\nits bus

- You can always pass filters yourself, built from the metadata that ingestion stored

In [38]:
docs = rag.retrieve("How did advertising revenue change in 2024 of Google?",
                    filters={"company_name": "alphabet inc."})

for d in docs:
    print(d.metadata["file_name"], "| page", d.metadata["page_number"])

google 10-k 2024.pdf | page 37
google 10-k 2024.pdf | page 33
google 10-k 2024.pdf | page 31
google 10-k 2024.pdf | page 38


- `extract_filters` shows what the LLM would pull from a question. With `auto_filter: true` the pipeline would apply this by itself; section 9 hands the decision to an agent instead

In [39]:
rag.extract_filters("How did Meta's advertising revenue change in 2024?")

{'company_name': 'meta', 'fiscal_year': 2024}

In [40]:
docs = rag.retrieve("What did Alphabet say about revenue growth in 2024?")

for d in docs:
    print(d.metadata["file_name"], "| page", d.metadata["page_number"])

google 10-k 2024.pdf | page 37
google 10-k 2024.pdf | page 36
google 10-k 2024.pdf | page 39
google 10-k 2024.pdf | page 31


## 7. Grounded answers with citations

- `query()` retrieves, numbers the sources, and instructs the model to answer from them alone
- Every claim must carry a citation. When the answer is a number, that is not optional

In [41]:
a = rag.query("What was Amazon's total net sales in 2024?")
print(a.formatted())

Amazon's total net sales in 2024 were $637,959 million (as reported in millions). This figure appears in the Consolidated Statements of Operations and the net sales breakdown by product/service group, both confirming the same total [1][2].

Sources:
  [1] data/amazon 10-k 2024.pdf
  [2] data/amazon 10-k 2024.pdf


In [42]:
a.citations

[Citation([1] data/amazon 10-k 2024.pdf),
 Citation([2] data/amazon 10-k 2024.pdf)]

## 8. The guardrail: refusing what the corpus does not contain

- Apple is not in this collection
- With no filters, retrieval still returns the closest chunks it has, from the wrong companies
- The chunks reach the model, and the grounding prompt is what refuses

In [45]:
b = rag.query("What was Apple's total revenue in 2024?")
b.refused

True

In [47]:
print(b.text)

The retrieved documents do not contain enough information to answer this question.


## 9. Agentic RAG: the model decides the filters

- Sections 6 to 8 chose filters by hand. An agent can make that decision itself
- Two tools: `get_filter_context` reports the metadata fields and the values the collection stores, `search_documents` runs retrieval with the filters the agent chose and labels each result with its file and page
- `create_agent` runs the loop: inspect the filters, search, answer
- Memory is a SQLite checkpointer on disk, so follow-up questions see the earlier turns

In [48]:
from typing import Optional

from langchain.tools import tool


@tool
def get_filter_context(query: str) -> str:
    """Get available metadata fields, stored values, and filter suggestions for a query.

    Call this before search_documents when the query involves specific metadata
    (company, year, document type, etc.). Use the returned context to decide
    what filters to pass to search_documents.
    """
    return rag.get_filter_context(query)


@tool
def search_documents(query: str, filters: Optional[dict] = None) -> str:
    """Search the document knowledge base for relevant information.

    Args:
        query: The search query
        filters: Optional metadata filters decided from get_filter_context.
                 Pass {} or omit to search without filtering.
    """
    results = rag.retrieve(query, top_k=5, filters=filters)
    if not results:
        return "No relevant documents found."

    chunks = []
    for doc in results:
        source = doc.metadata.get("file_name", "unknown")
        page = doc.metadata.get("page_number")
        if page is not None:
            source = source + ", page " + str(page)
        chunks.append("[" + source + "]\n" + doc.page_content)

    return "\n\n---\n\n".join(chunks)

- Same code in `scripts/tools.py`, so the Chainlit app can hand the same tools to its own agent

In [49]:
from scripts.tools import get_filter_context, search_documents

In [50]:
import sqlite3
from pathlib import Path

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from langgraph.checkpoint.sqlite import SqliteSaver

model = ChatOllama(model="nemotron-3.5-lightning", base_url="http://localhost:11434")

Path("db").mkdir(exist_ok=True)
Path("db/agent_memory.db").unlink(missing_ok=True)   # start the demo on a clean thread
conn = sqlite3.connect("db/agent_memory.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)

agent = create_agent(
    model=model,
    tools=[get_filter_context, search_documents],
    system_prompt=(
        "You are a helpful financial document assistant. "
        "For complex questions, break them down into simple sub-questions and answer each one before forming a final answer. "
        "Always use search_documents to retrieve information before answering. "
        "Never answer from general knowledge. "
        "Use get_filter_context before search_documents when the query involves specific metadata (company, year, document type, etc.). "
        "When a question asks about the whole collection, or for the highest, lowest, or a comparison across companies, first call get_filter_context to see which companies the collection holds, then search every relevant company before answering. "
        "If no relevant documents are found, say so. Do not guess or fabricate an answer. "
        "Always cite the source document in your answer."
    ),
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "class-demo"}}

- `agent.invoke` waits in silence and hands back the end state. `agent.stream` can surface every step as it happens instead
- `stream_agent` wraps it in a generator: it always yields the answer, and by default it also yields every tool call and result, so you watch the agent decide. `verbose=False` gives the answer alone
- The events are plain tuples, so `print(*event)` is enough to watch it work. Tool results are truncated, the stream is for display

In [51]:
def stream_agent(agent, query, config, verbose=True):
    """Run one query and yield the agent's progress as it happens."""
    for event in agent.stream({"messages": [HumanMessage(query)]},
                              config=config, stream_mode="updates"):
        for update in event.values():
            for message in update.get("messages", []):
                if message.type == "ai" and message.tool_calls:
                    if verbose:
                        for call in message.tool_calls:
                            yield "tool_call", call["name"], call["args"]
                elif message.type == "tool":
                    if verbose:
                        yield "tool_result", message.name, message.content[:200]
                elif message.type == "ai" and message.content:
                    yield "answer", message.content

- Same code in `scripts/agent.py`, so the notebook and the Chainlit app run one implementation

In [52]:
from scripts.agent import stream_agent

In [53]:
for event in stream_agent(agent, "What was Google's advertising revenue in 2024?",
                          config):
    print(*event)

tool_call get_filter_context {'query': 'Google advertising revenue 2024'}
tool_result get_filter_context ## RAGWire Filter Context

### Available Metadata Fields and Stored Values
- **company_name**: ['alphabet inc.', 'amazon.com, inc.']
- **doc_type**: ['10-k']
- **fiscal_quarter**: []
- **fiscal_year**
tool_call search_documents {'query': 'Google advertising revenue 2024', 'filters': {'company_name': 'alphabet inc.', 'fiscal_year': 2024, 'doc_type': '10-k'}}
tool_result search_documents [google 10-k 2024.pdf, page 37]
Table of Contents Alphabet Inc.
Financial Results
Revenues
The following table presents revenues by type (in millions):
Year Ended December 31,
2023 2024
Google Search 
answer **Google's advertising revenue in 2024 was $264,590 million (about $264.6 billion).**  

This figure comes from Alphabet Inc.'s 2024 Form 10‑K annual report, which lists “Google advertising” revenues as **$264,590 million** for the year ended December 31 2024【google 10‑k 2024.pdf, page 37】. The sa

- The follow-up only makes sense if the agent remembers the last answer

In [54]:
for event in stream_agent(agent, "How does that compare to Amazon's total revenue in 2024?",
                          config):
    print(*event)

tool_call get_filter_context {'query': 'Amazon total revenue 2024'}
tool_result get_filter_context ## RAGWire Filter Context

### Available Metadata Fields and Stored Values
- **company_name**: ['alphabet inc.', 'amazon.com, inc.']
- **doc_type**: ['10-k']
- **fiscal_quarter**: []
- **fiscal_year**
tool_call search_documents {'query': 'Amazon total revenue 2024', 'filters': {'company_name': 'amazon.com, inc.', 'doc_type': '10-k', 'fiscal_year': 2024}}
tool_result search_documents [amazon 10-k 2024.pdf, page 43]
Table of Contents
Revenue
Revenue is measured based on the amount of consideration that we expect to receive, reduced by estimates for return allowances, promotional di
answer **Comparison of Google’s 2024 advertising revenue with Amazon’s 2024 total revenue**

| Company | Revenue figure (2024) | Source |
|---------|----------------------|--------|
| **Google (Alphabet Inc.) – advertising revenue** | **$264,590 million** (≈ $264.6 billion) | *Alphabet 2024 Form 10‑K*, “Google ad

- A third question spans the whole collection, so the prompt rule sends the agent through every company before it answers

In [55]:
for event in stream_agent(agent, "Which company in the collection had the highest total revenue in 2024?",
                          config):
    print(*event)

answer **Highest total revenue in 2024 among the companies in the collection**

| Company (fiscal year 2024) | Total revenue (mill $) | Source |
|----------------------------|------------------------|--------|
| **Amazon.com, Inc.** | **$637,959 million** (≈ $638 billion) | Amazon 2024 Form 10‑K, Consolidated Statements of Operations – “Total net sales” for the year ended Dec 31 2024 (page 37)【amazon 10‑k 2024.pdf, page 37】 |
| **Alphabet Inc. (Google)** | $350,018 million (≈ $350 billion) | Alphabet 2024 Form 10‑K, “Total revenues” for the year ended Dec 31 2024 (page 37)【google 10‑k 2024.pdf, page 37】 |

### Interpretation
- The two companies in the dataset are **Amazon.com, Inc.** and **Alphabet Inc.** (Google).  
- Amazon’s **total revenue** of **$637.96 billion** exceeds Google’s **total revenue** of **$350.02 billion** by roughly **$287.9 billion** (about 82 % more).  
- Therefore, **Amazon had the highest total revenue in 2024** within this collection.

> **Bottom line:** Amazon

- The checkpointer holds the whole thread. `get_state` reads it back without running the agent

In [56]:
messages = agent.get_state(config).values["messages"]
[type(m).__name__ for m in messages]

['HumanMessage',
 'AIMessage',
 'ToolMessage',
 'AIMessage',
 'ToolMessage',
 'AIMessage',
 'HumanMessage',
 'AIMessage',
 'ToolMessage',
 'AIMessage',
 'ToolMessage',
 'AIMessage',
 'HumanMessage',
 'AIMessage']

In [57]:
messages

[HumanMessage(content="What was Google's advertising revenue in 2024?", additional_kwargs={}, response_metadata={}, id='83905e85-94df-4ca5-af40-8ce0edb359aa'),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'nemotron-3.5-lightning', 'created_at': '2026-08-14T16:27:10.8776243Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9255297900, 'load_duration': 8237917600, 'prompt_eval_count': 582, 'prompt_eval_duration': 332324000, 'eval_count': 114, 'eval_duration': 679650000, 'logprobs': None, 'model_name': 'nemotron-3.5-lightning', 'model_provider': 'ollama'}, id='lc_run--01a00119-2a74-74d2-914a-8466b9270014-0', tool_calls=[{'name': 'get_filter_context', 'args': {'query': 'Google advertising revenue 2024'}, 'id': '55cafab9-3e05-43ed-a588-7a592b0e9e96', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 582, 'output_tokens': 114, 'total_tokens': 696}),
 ToolMessage(content="## RAGWire Filter Context\n\n### Available Metadata Fields

## 10. From notebook to app

- `scripts/pipeline.py` holds the finalized pipeline, `get_rag()`; `scripts/tools.py` builds the tools on top of it
- `app.py` is a Chainlit UI over the agent you just built: the same tools, the same SQLite memory, and the same `stream_agent`, with the tool events drawn as steps in the chat
- Each chat session gets its own memory thread
- Run it from a terminal:

```
uv run chainlit run app.py
```

## 11. Inspecting the MCP server

- RAGWire ships an MCP server over stdio: `ragwire mcp serve`
- MCP Inspector is the standard debugging UI: it launches the server as a child process, lists its tools, and lets you call them by hand
- Needs Node.js; run it from this folder:

```
npx -y @modelcontextprotocol/inspector -e LANGSMITH_TRACING=false -- uv run ragwire mcp serve --config config/finance_rag.yaml
```

- The browser opens on the inspector UI: Connect, then List Tools
- `search_documents`, `get_filter_context` and `collection_stats` return quickly; `answer_question` runs the LLM, about ten seconds warm
- `LANGSMITH_TRACING=false` is not optional: the tracer shells out to git for run metadata, and inside a stdio MCP server that subprocess never returns

## 12. The collection as a tool: MCP

- RAGWire ships an MCP server: `search_documents`, `answer_question`, `get_filter_context`, and `collection_stats` become tools Claude can call
- Add it to Claude in `claude_desktop_config.json`:

```
{
  "mcpServers": {
    "finance-rag": {
      "command": "D:\\Corporate Training\\2026\\finance-rag-bootcamp\\.venv\\Scripts\\ragwire.exe",
      "args": ["mcp", "serve", "--config",
               "D:\\Corporate Training\\2026\\finance-rag-bootcamp\\config\\finance_rag.yaml"],
      "env": {"LANGSMITH_TRACING": "false"}
    }
  }
}
```